# 7. Results Visualization & Evaluation

## Overview
This notebook evaluates the optimized networks produced by the Genetic Algorithm (Notebook 06) against the real-world baseline networks.

### Evaluation Framework
The **fitness function** combines five normalized metrics with the following weights:

| Metric | Weight | Interpretation |
|--------|--------|---------------|
| Average Distance | 30% | Mean road distance between connected stations |
| Diameter | 10% | Worst-case shortest path length |
| Clustering Coefficient | 10% | Local connectivity patterns |
| Network Density | 10% | Ratio of actual to possible edges |
| Coverage Area Ratio | 40% | Network convex hull vs. country area |

**Lower fitness = better network** (minimization problem)

In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import sys
from shapely.geometry import Point, MultiPoint

sys.path.insert(0, '../src')
from utils import get_osrm_distance, draw_graph, calculate_weighted_metrics, weighted_mean

## 7.1 Load Data and Define Fitness Function

In [ ]:
# Load Germany's boundary polygon
germany_boundary = gpd.read_file('../data/boundaries/4_niedrig.geo.json')
germany_polygon = germany_boundary.geometry.unary_union

def calculate_network_to_country_area_ratio(graph, country_polygon):
    """Compute the ratio of the network's convex hull area to the country's area."""
    node_coords = [
        (graph.nodes[n]['longitude'], graph.nodes[n]['latitude'])
        for n in graph.nodes()
    ]
    network_hull = MultiPoint(node_coords).convex_hull
    return network_hull.area / country_polygon.area

def fitness_function(graph, year, max_distance=100):
    """Compute the multi-objective fitness score for a network."""
    metrics = calculate_weighted_metrics(graph, year)
    coverage = calculate_network_to_country_area_ratio(graph, germany_polygon)

    # Min-max normalization using observed historical bounds
    norm_avg_dist = (metrics['average_distance'] - 15.10) / (58.43 - 15.20)
    norm_diameter = (metrics['diameter'] - 1) / (11 - 1)
    norm_clustering = (metrics['average_clustering'] - 0.72) / (0.93 - 0.72)
    norm_density = (metrics['density'] - 0.08) / (1 - 0.08)

    weights = [0.3, 0.1, 0.1, 0.1, 0.4]
    values = [
        norm_avg_dist,
        norm_diameter,
        norm_clustering,
        1 - norm_density,
        1 - coverage
    ]
    return weighted_mean(values, weights)

## 7.2 Compare Baseline vs. Optimized Networks

Load both the real-world (baseline) networks and GA-optimized networks for each year, then compare their fitness scores and topology metrics.

In [ ]:
# Evaluate baseline (real-world) networks
eval_years = [2011, 2012, 2013, 2014, 2015, 2016, 2017]

print('=== Baseline Network Fitness Scores ===')
for yr in eval_years:
    graph = nx.read_graphml(f'../results/networks/baseline/network_{yr}_100.graphml')
    fit = fitness_function(graph, yr)
    metrics = calculate_weighted_metrics(graph, yr)
    print(f'  {yr}: fitness={fit:.4f}, nodes={metrics["total_nodes"]}, '
          f'density={metrics["density"]:.4f}, avg_dist={metrics["average_distance"]:.2f} km')

## 7.3 Visualize Optimized Networks

Compare the visual structure of optimized vs. baseline networks.

In [ ]:
# Load and visualize an optimized network
opt_year = 2012
graph_opt = nx.read_graphml(f'../results/networks/optimized/network_{opt_year}_optimized.graphml')
metrics_opt = calculate_weighted_metrics(graph_opt, opt_year)
print(f'Optimized network ({opt_year}): {metrics_opt}')
draw_graph(graph_opt, title=f'Optimized EV Charging Network ({opt_year})')

In [ ]:
# Compare with the baseline network for the same year
graph_baseline = nx.read_graphml(f'../results/networks/baseline/network_{opt_year}_100.graphml')
metrics_baseline = calculate_weighted_metrics(graph_baseline, opt_year)
print(f'Baseline network ({opt_year}): {metrics_baseline}')
draw_graph(graph_baseline, title=f'Baseline EV Charging Network ({opt_year})')